# API-Sports.io NFL Stats - Automated Daily Update

Automatically fetches missing weeks from API-Sports.io with smart quota management.

**Schedule:** Runs **DAILY** at 2:00 AM (America/Chicago)

**Rate Limit Management:**
- ⚠️ **100 API requests/day limit** (free tier)
- ~33 requests needed per week (16 games + overhead)
- Fetches ONE missing week per day
- Gradually backfills all weeks without hitting limits
- Tracks API usage and stops if quota is low

**Strategy:**
1. Check what weeks we already have
2. Find the earliest missing week in current season
3. Verify we have enough API quota (50+ requests remaining)
4. Fetch that week's data
5. Track usage and exit

**Source:** API-Sports.io NFL API

In [0]:
import requests
import json
import time
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from datetime import datetime

# API-Sports.io Configuration
API_KEY = "ba394c1e6cd4f63490c23b32430d1f3c"
BASE_URL = "https://v1.american-football.api-sports.io"

headers = {"x-apisports-key": API_KEY}

print("✓ API-Sports.io configured")
print(f"✓ Free tier: 100 requests/day")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
# Historical backfill for 2024, 2025, and 2026 with smart quota management
# Run this multiple times over several days to backfill all historical data
# Each run fetches 2-3 weeks depending on available API quota

import time

print("="*70)
print("API-SPORTS.IO - HISTORICAL BACKFILL (RATE LIMITED)")
print("="*70)
print("\n⚠️  100 API requests/day limit - fetches 2-3 weeks per run")
print("🔁 Run this multiple times over several days to backfill all data")
print("\nThis is a one-time backfill - use the 'Fetch Latest Week Only' cell for scheduled updates\n")

# Determine target seasons
backfill_seasons = [2024, 2025, 2026]  # Updated to include 2026

# Check API quota first
print("--- STEP 1: Check API Quota ---")
try:
    test_response = requests.get(
        f"{BASE_URL}/games",
        headers=headers,
        params={"league": "1", "season": "2024"},
        timeout=30
    )
    
    requests_remaining = int(test_response.headers.get('x-ratelimit-requests-remaining', 0))
    requests_limit = int(test_response.headers.get('x-ratelimit-requests-limit', 100))
    
    print(f"✓ API quota check successful")
    print(f"  Daily limit: {requests_limit} requests")
    print(f"  Remaining:   {requests_remaining} requests")
    print(f"  Used today:  {requests_limit - requests_remaining} requests")
    
    # Need at least 40 requests to safely fetch 1 week (~33 per week + overhead)
    MIN_REQUIRED = 40
    if requests_remaining < MIN_REQUIRED:
        print(f"\n⚠️  Insufficient API quota: {requests_remaining} < {MIN_REQUIRED} required")
        print("Exiting to preserve quota. Try again tomorrow.")
        dbutils.notebook.exit(f"Insufficient quota: {requests_remaining} remaining")
    
    # Calculate how many weeks we can fetch (conservative estimate)
    max_weeks_to_fetch = min(3, (requests_remaining - 10) // 35)  # 35 requests per week, keep 10 buffer
    print(f"\n✓ Can safely fetch ~{max_weeks_to_fetch} weeks with current quota\n")
    
except Exception as e:
    print(f"❌ Error checking API quota: {e}")
    raise

api_calls_made = 1

# Check what weeks we already have across all seasons
print("--- STEP 2: Find Missing Weeks ---")
all_missing_weeks = []

for season in backfill_seasons:
    existing_weeks = spark.sql(f"""
        SELECT DISTINCT week 
        FROM main.fantasai.silver_weekly_stats 
        WHERE season = {season} AND source = 'api_sports'
        ORDER BY week
    """).collect()
    
    existing_week_numbers = [row['week'] for row in existing_weeks]
    all_weeks = set(range(1, 19))
    missing_weeks = sorted(all_weeks - set(existing_week_numbers))
    
    if existing_week_numbers:
        print(f"Season {season}: Have weeks {existing_week_numbers}")
    else:
        print(f"Season {season}: No data yet (expected if season hasn't started)")
    
    if missing_weeks:
        print(f"  Missing: {missing_weeks}")
        for week in missing_weeks:
            all_missing_weeks.append((season, week))
    else:
        print(f"  ✓ Complete!")

if not all_missing_weeks:
    print(f"\n✓ All weeks (1-18) already have data for {', '.join(map(str, backfill_seasons))}!")
    print("No missing data to fetch.")
    dbutils.notebook.exit("All weeks up to date")

print(f"\nTotal missing weeks: {len(all_missing_weeks)}")
print(f"Will fetch: {min(max_weeks_to_fetch, len(all_missing_weeks))} weeks this run\n")

# Fetch missing weeks (up to our quota limit)
weeks_to_fetch = all_missing_weeks[:max_weeks_to_fetch]
print("--- STEP 3: Fetch Data ---")

total_records = 0
successful_weeks = []
failed_weeks = []

for season, week in weeks_to_fetch:
    try:
        print(f"\n[Season {season}, Week {week}] Fetching...")
        
        # Fetch all games for this season (we already did this for 2024, so cache it)
        if season == 2024 and api_calls_made == 1:
            all_games = test_response.json().get('response', [])
        else:
            games_response = requests.get(
                f"{BASE_URL}/games",
                headers=headers,
                params={"league": "1", "season": str(season)},
                timeout=30
            )
            api_calls_made += 1
            all_games = games_response.json().get('response', [])
        
        week_name = f"Week {week}"
        week_games = [g for g in all_games if g.get('game', {}).get('week') == week_name]
        
        if not week_games:
            print(f"  ⚠️  No games found - skipping (expected if week hasn't been played yet)")
            failed_weeks.append((season, week, "No games"))
            continue
        
        game_ids = [g.get('game', {}).get('id') for g in week_games if g.get('game', {}).get('id')]
        print(f"  Found {len(game_ids)} games")
        
        # Check if we have enough quota for this week
        if len(game_ids) > requests_remaining - api_calls_made - 10:
            print(f"  ⚠️  Insufficient quota for this week ({len(game_ids)} requests needed)")
            print(f"  Stopping here to preserve quota. Resume on next run.")
            break
        
        # Fetch player stats for each game
        all_player_records = []
        
        for idx, game_id in enumerate(game_ids, 1):
            game_stats_response = requests.get(
                f"{BASE_URL}/games/statistics/players",
                headers=headers,
                params={"id": str(game_id)},
                timeout=30
            )
            api_calls_made += 1
            
            if game_stats_response.status_code == 200:
                game_stats = game_stats_response.json().get('response', [])
                
                for team_data in game_stats:
                    team_name = team_data.get('team', {}).get('name')
                    groups = team_data.get('groups', [])
                    
                    for group in groups:
                        group_name = group.get('name')
                        players = group.get('players', [])
                        
                        for player_entry in players:
                            player_info = player_entry.get('player', {})
                            player_id = player_info.get('id')
                            player_name = player_info.get('name')
                            
                            stats_array = player_entry.get('statistics', [])
                            stats_dict = {'stat_group': group_name}
                            for stat in stats_array:
                                stat_name = stat.get('name', '').lower().replace(' ', '_')
                                stat_value = stat.get('value')
                                stats_dict[stat_name] = stat_value
                            
                            all_player_records.append({
                                'player_id': str(player_id),
                                'player_name': player_name,
                                'team': team_name,
                                'stat_group': group_name,
                                'statistics': stats_dict,
                                'game_id': str(game_id)
                            })
            
            time.sleep(0.2)
        
        if not all_player_records:
            print(f"  ⚠️  No player data extracted - skipping")
            failed_weeks.append((season, week, "No player data"))
            continue
        
        # Convert to DataFrame
        rows = []
        for record in all_player_records:
            rows.append(
                Row(
                    player_id=record['player_id'],
                    week=int(week),
                    season=int(season),
                    fantasy_points=0.0,
                    stats=json.dumps(record),
                    source='api_sports'
                )
            )
        
        stats_df = spark.createDataFrame(rows)
        
        # Write to bronze
        bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())
        bronze_df.createOrReplaceTempView("api_sports_backfill_bronze")
        
        spark.sql("""
            MERGE INTO main.fantasai.bronze_weekly_stats AS target
            USING api_sports_backfill_bronze AS source
            ON target.player_id = source.player_id 
                AND target.week = source.week 
                AND target.season = source.season
                AND target.source = source.source
            WHEN MATCHED THEN
                UPDATE SET
                    target.fantasy_points = source.fantasy_points,
                    target.stats = source.stats,
                    target.ingested_at = source.ingested_at
            WHEN NOT MATCHED THEN
                INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
                VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
        """)
        
        # Write to silver
        silver_df = bronze_df.groupBy("player_id", "week", "season", "source").agg(
            F.first("fantasy_points").alias("fantasy_points"),
            F.first("stats").alias("stats"),
            F.max("ingested_at").alias("ingested_at")
        )
        
        silver_df.createOrReplaceTempView("api_sports_backfill_silver")
        
        spark.sql("""
            MERGE INTO main.fantasai.silver_weekly_stats AS target
            USING api_sports_backfill_silver AS source
            ON target.player_id = source.player_id 
                AND target.week = source.week 
                AND target.season = source.season
                AND target.source = source.source
            WHEN MATCHED THEN
                UPDATE SET
                    target.fantasy_points = source.fantasy_points,
                    target.stats = source.stats,
                    target.ingested_at = source.ingested_at
            WHEN NOT MATCHED THEN
                INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
                VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
        """)
        
        total_records += len(rows)
        successful_weeks.append((season, week))
        print(f"  ✓ Success! {len(rows)} player records ingested")
        
    except Exception as e:
        print(f"  ❌ Error: {str(e)}")
        failed_weeks.append((season, week, str(e)))

# Summary
print("\n" + "="*70)
print("BACKFILL SUMMARY")
print("="*70)
print(f"\nAPI calls made: {api_calls_made}")
print(f"Total player records: {total_records}")
print(f"\nSuccessful weeks ({len(successful_weeks)}):")
for season, week in successful_weeks:
    print(f"  ✓ Season {season}, Week {week}")

if failed_weeks:
    print(f"\nFailed/Skipped weeks ({len(failed_weeks)}):")
    for season, week, reason in failed_weeks:
        print(f"  ⚠️  Season {season}, Week {week}: {reason}")

print(f"\n{'='*70}")

if len(all_missing_weeks) > len(successful_weeks):
    remaining = len(all_missing_weeks) - len(successful_weeks)
    print(f"\n📌 {remaining} weeks still remaining - run this cell again tomorrow")
else:
    print(f"\n✓ All available weeks have been fetched!")
    print(f"   Note: 2026 data will become available during the 2026 season")

In [0]:
# Fetch only the latest week's data with smart quota management
# Use this cell for daily scheduled jobs

import time

print("="*70)
print("API-SPORTS.IO - LATEST DATA FETCH")
print("="*70)
print(f"\nCurrent date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Determine current season
current_date = datetime.now()
current_year = current_date.year
current_month = current_date.month

if current_month >= 9:  # September onwards = current year's season
    CURRENT_SEASON = current_year
else:  # January-August = previous year's season
    CURRENT_SEASON = current_year - 1

print(f"Detected NFL season: {CURRENT_SEASON}")

# Check API quota first
print("\n--- STEP 1: Check API Quota ---")
try:
    test_response = requests.get(
        f"{BASE_URL}/games",
        headers=headers,
        params={"league": "1", "season": str(CURRENT_SEASON)},
        timeout=30
    )
    
    requests_remaining = int(test_response.headers.get('x-ratelimit-requests-remaining', 0))
    requests_limit = int(test_response.headers.get('x-ratelimit-requests-limit', 100))
    
    print(f"✓ API quota check successful")
    print(f"  Daily limit: {requests_limit} requests")
    print(f"  Remaining:   {requests_remaining} requests")
    print(f"  Used today:  {requests_limit - requests_remaining} requests")
    
    # We need ~33 requests per week, so require at least 40 to be safe
    MIN_REQUIRED = 40
    if requests_remaining < MIN_REQUIRED:
        print(f"\n⚠️  Insufficient API quota: {requests_remaining} < {MIN_REQUIRED} required")
        print("Exiting to preserve quota. Will retry tomorrow.")
        dbutils.notebook.exit(f"Insufficient quota: {requests_remaining} remaining")
    
    print(f"\n✓ Sufficient quota available ({requests_remaining} >= {MIN_REQUIRED})\n")
    
except Exception as e:
    print(f"❌ Error checking API quota: {e}")
    raise

api_calls_made = 1

# Find the latest week with completed games (check backwards from week 18)
print("--- STEP 2: Auto-Detect Latest Week ---")
latest_week = None
all_games = test_response.json().get('response', [])

for check_week in range(18, 0, -1):
    week_name = f"Week {check_week}"
    week_games = [g for g in all_games if g.get('game', {}).get('week') == week_name]
    
    if week_games:
        # Check if games have been played (not just scheduled)
        completed_count = sum(1 for g in week_games if g.get('game', {}).get('status', {}).get('short') in ['FT', 'F/OT'])
        
        if completed_count > 0:
            latest_week = check_week
            print(f"✓ Found latest week with completed games: Week {latest_week}")
            print(f"  ({completed_count}/{len(week_games)} games completed)")
            break

if not latest_week:
    print("⚠️  No completed games found - season may not have started yet")
    print("Defaulting to Week 1")
    latest_week = 1

WEEK = latest_week
print(f"\nWill fetch: Season {CURRENT_SEASON}, Week {WEEK}\n")

try:
    # Get games for this week
    print(f"--- STEP 3: Fetch Week {WEEK} Data ---")
    
    week_name = f"Week {WEEK}"
    week_games = [g for g in all_games if g.get('game', {}).get('week') == week_name]
    
    if not week_games:
        print(f"\n⚠️  No games found for Week {WEEK}")
        print("Week may not have been played yet or data not available")
        dbutils.notebook.exit(f"No games available for Season {CURRENT_SEASON} Week {WEEK}")
    
    game_ids = [g.get('game', {}).get('id') for g in week_games if g.get('game', {}).get('id')]
    print(f"✓ Found {len(game_ids)} games")
    print(f"\nEstimated API calls needed: {len(game_ids)} (one per game)")
    print(f"Current API quota remaining: {requests_remaining}\n")
    
    if len(game_ids) > requests_remaining - 10:  # Keep 10 as buffer
        print(f"⚠️  Not enough quota to safely fetch all games")
        print(f"Need {len(game_ids)} requests, have {requests_remaining} remaining")
        dbutils.notebook.exit("Insufficient quota for this week")
    
    # Fetch player stats for each game
    print(f"Fetching player stats for {len(game_ids)} games...\n")
    all_player_records = []
    
    for idx, game_id in enumerate(game_ids, 1):
        game_stats_response = requests.get(
            f"{BASE_URL}/games/statistics/players",
            headers=headers,
            params={"id": str(game_id)},
            timeout=30
        )
        api_calls_made += 1
        
        if game_stats_response.status_code == 200:
            game_stats = game_stats_response.json().get('response', [])
            
            # Parse nested structure: team → groups → players
            players_count = 0
            for team_data in game_stats:
                team_name = team_data.get('team', {}).get('name')
                groups = team_data.get('groups', [])
                
                for group in groups:
                    group_name = group.get('name')
                    players = group.get('players', [])
                    
                    for player_entry in players:
                        player_info = player_entry.get('player', {})
                        player_id = player_info.get('id')
                        player_name = player_info.get('name')
                        
                        # Convert statistics array to dictionary
                        stats_array = player_entry.get('statistics', [])
                        stats_dict = {'stat_group': group_name}
                        for stat in stats_array:
                            stat_name = stat.get('name', '').lower().replace(' ', '_')
                            stat_value = stat.get('value')
                            stats_dict[stat_name] = stat_value
                        
                        all_player_records.append({
                            'player_id': str(player_id),
                            'player_name': player_name,
                            'team': team_name,
                            'stat_group': group_name,
                            'statistics': stats_dict,
                            'game_id': str(game_id)
                        })
                        players_count += 1
            
            print(f"  Game {idx}/{len(game_ids)}: +{players_count} records")
        else:
            print(f"  Game {idx}/{len(game_ids)}: Failed ({game_stats_response.status_code})")
        
        # Small delay to be respectful to API
        time.sleep(0.2)
    
    print(f"\n✓ Total player records: {len(all_player_records)}")
    print(f"✓ Total API calls made: {api_calls_made}\n")
    
    if not all_player_records:
        print("⚠️  No player data extracted")
        dbutils.notebook.exit(f"No player data for Season {CURRENT_SEASON} Week {WEEK}")
    
    # Convert to DataFrame
    rows = []
    for record in all_player_records:
        rows.append(
            Row(
                player_id=record['player_id'],
                week=int(WEEK),
                season=int(CURRENT_SEASON),
                fantasy_points=0.0,
                stats=json.dumps(record),
                source='api_sports'
            )
        )
    
    stats_df = spark.createDataFrame(rows)
    
    # Write to bronze
    bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())
    bronze_df.createOrReplaceTempView("api_sports_bronze_temp")
    
    spark.sql("""
        MERGE INTO main.fantasai.bronze_weekly_stats AS target
        USING api_sports_bronze_temp AS source
        ON target.player_id = source.player_id 
            AND target.week = source.week 
            AND target.season = source.season
            AND target.source = source.source
        WHEN MATCHED THEN
            UPDATE SET
                target.fantasy_points = source.fantasy_points,
                target.stats = source.stats,
                target.ingested_at = source.ingested_at
        WHEN NOT MATCHED THEN
            INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
            VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    # Write to silver (deduplicate by player since they can appear in multiple stat groups)
    silver_df = bronze_df.groupBy("player_id", "week", "season", "source").agg(
        F.first("fantasy_points").alias("fantasy_points"),
        F.first("stats").alias("stats"),
        F.max("ingested_at").alias("ingested_at")
    )
    
    silver_df.createOrReplaceTempView("api_sports_silver_temp")
    
    spark.sql("""
        MERGE INTO main.fantasai.silver_weekly_stats AS target
        USING api_sports_silver_temp AS source
        ON target.player_id = source.player_id 
            AND target.week = source.week 
            AND target.season = source.season
            AND target.source = source.source
        WHEN MATCHED THEN
            UPDATE SET
                target.fantasy_points = source.fantasy_points,
                target.stats = source.stats,
                target.ingested_at = source.ingested_at
        WHEN NOT MATCHED THEN
            INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
            VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    player_count = silver_df.count()
    
    print("="*70)
    print("FETCH COMPLETE")
    print("="*70)
    print(f"\n✓ Season: {CURRENT_SEASON}")
    print(f"✓ Week: {WEEK}")
    print(f"✓ Unique players stored: {player_count}")
    print(f"✓ Total records: {len(all_player_records)}")
    print(f"✓ API calls used: {api_calls_made}")
    print(f"\n📅 Data ingested at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\n💡 Run the 'Historical Backfill' cell manually to fetch older weeks")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    print(f"API calls made before error: {api_calls_made}")
    import traceback
    traceback.print_exc()
    raise